# Integração da LLM com dados estruturados e protocolos

Este notebook mostra como uma pergunta recebe contexto antes de chegar ao modelo. A aplicação consulta dados estruturados do paciente, procura protocolos relacionados e reúne essas informações em um prompt. A finalidade é limitar a geração às informações disponíveis na execução atual, em vez de pedir uma resposta baseada apenas no conhecimento geral do modelo.

O LangChain é usado como um encadeamento explícito: uma etapa prepara os dados de entrada e outra chama o gerador. Essa separação torna visível quais informações participaram da resposta e impede que o modelo escolha comandos SQL ou altere registros.

O notebook apresenta separadamente a consulta do paciente, a recuperação dos protocolos e a geração bruta. O controle final de segurança é mostrado no notebook seguinte.

In [1]:
from pathlib import Path
import json, os, sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*", module="tqdm.auto")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ["HF_HOME"] = str(ROOT / ".hf-cache")
import subprocess
subprocess.run([sys.executable, "scripts/init_database.py"], check=True)
from clinical_assistant.data_access import ClinicalRepository
patient = ClinicalRepository("data/processed/hospital.db").get_patient_context("PAC-0001")
print(patient.as_prompt_context())

Identificador institucional: PAC-0001
Ano de nascimento: 1968; sexo registrado: F
Condições registradas: hipertensão;diabetes tipo 2; alergias: penicilina
Exames pendentes: creatinina (solicitado em 2026-07-20); eletrocardiograma (solicitado em 2026-07-27)
Resultados recentes: hemoglobina glicada: 7.8% (2026-06-20)


## Consulta estruturada: contexto por paciente

A primeira célula cria o SQLite quando necessário e consulta um paciente sintético por identificador. O repositório retorna somente campos definidos para a aplicação: condições registradas, alergias, exames pendentes e resultados. Cada execução lê o estado atual do arquivo local, portanto essas informações não ficam gravadas dentro do prompt de treinamento.

O banco armazena dados em campos específicos e é acessado por consultas controladas e somente de leitura. O modelo recebe apenas um resumo desses campos para produzir um rascunho. Ele não pode inserir, atualizar ou apagar dados.

Os registros exibidos são fictícios. A consulta demonstra a integração técnica e não uma conexão com prontuário hospitalar.

In [2]:
from clinical_assistant.retrieval import ProtocolRetriever
retriever = ProtocolRetriever("data/raw/protocols")
sources = retriever.retrieve("Exames de acompanhamento de diabetes", k=2, minimum_score=0.08)
for source in sources:
    print(source.source_id, source.version, source.score)
    print(source.excerpt)

PROTO-DIABETES 1.0 0.3279
# PROTO-DIABETES — Acompanhamento de diabetes em consulta

**Versão sintética:** 1.0 — **revisão:** 2026-02-20
PROTO-HIPERTENSAO 1.0 0.128
## Verificações de acompanhamento

- confirmar técnica e repetição da medida quando houver valor isolado discrepante;
- revisar medidas anteriores, adesão relatada, sintomas e medicamentos registrados;
- conferir função renal e eletrólitos quando esses exames constarem no plano assistencial;
- organizar fatores de risco e pendências para discussão clínica.


## Recuperação de protocolos: seleção de contexto

Nem todo protocolo disponível é relevante para toda pergunta. A recuperação lexical procura palavras em comum entre a pergunta e os documentos, ordenando aqueles que têm maior proximidade textual. O objetivo não é provar que um documento é suficiente ou verdadeiro, mas restringir o contexto fornecido ao modelo a fontes relacionadas ao assunto.

A aplicação retorna até dois protocolos que superam um limiar mínimo. Para cada fonte, mostra código, versão, pontuação e trecho selecionado. A pontuação indica proximidade de texto, não probabilidade de correção. O trecho também tem tamanho limitado para que um documento não ocupe todo o contexto.

Esse método depende das palavras da pergunta. Sinônimos, abreviações e termos ausentes dos protocolos podem impedir a recuperação esperada. Quando não há fontes, o fluxo retém a resposta em vez de gerar texto livre.

In [3]:
from clinical_assistant.llm import T5Generator
from clinical_assistant.chains import build_clinical_chain
generator = T5Generator("models/clinical-t5-lora")
chain = build_clinical_chain(generator)
draft = chain.invoke({"question": "Exames de acompanhamento de diabetes",
    "patient_context": patient.as_prompt_context(),
    "protocol_context": "\n\n".join(s.excerpt for s in sources)})
print("Saída bruta do adaptador:", draft)

Saída bruta do adaptador: Representaçes de acompanhamento - confirmar técnica e repetiço da medida cuando houver valor isolado discrepante; - revisar medidas anteriores, adeso relatada, sintomas e medicamentos registrados; - conferir funcionarios renal e eletrólitos cuando escritos constarem no plano assistencial; - organizar fatores de risco e pendências para discutir clnica. Resposta:


## Geração bruta para inspeção

A última célula carrega o modelo base e o adaptador LoRA salvo localmente. O prompt reúne pergunta, resumo do paciente e trechos recuperados. A saída é exibida sem bloqueio para que seja possível observar o comportamento real do modelo.

Separar geração e validação impede que uma mensagem de segurança esconda falhas como repetição, ausência de contexto ou conteúdo impróprio. O notebook seguinte recebe esse rascunho dentro do fluxo de decisão e determina se ele pode ser apresentado para revisão.

A presença de protocolos no prompt não garante interpretação correta. Por isso, a geração nunca é tratada como autorização para definir conduta.